In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
import pickle

# Load data
churn_data = pd.read_csv('churn_data_with_features.csv')

print("="*70)
print("PREDICTIVE MODELING: Churn Prediction with Competitor Threat")
print("="*70)

# SECTION 1: Prepare Features
segment_map = {'Budget': 1, 'Mid-tier': 2, 'Premium': 3, 'Enterprise': 4}

X = churn_data[[
    'individual_competitor_threat',
    'tenure',
    'MonthlyCharges',
    'TotalCharges'
]].fillna(0).copy()

X['segment_numeric'] = churn_data['segment'].map(segment_map).fillna(2)
X = X[['individual_competitor_threat', 'tenure', 'MonthlyCharges', 'TotalCharges', 'segment_numeric']]

y = churn_data['Churn_binary']

print(f"\n✅ Features prepared: X shape {X.shape}, y shape {y.shape}")

# SECTION 2: Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"  Train: {X_train.shape}, Test: {X_test.shape}")

# SECTION 3: Logistic Regression
print(f"\n{'='*70}")
print("MODEL 1: LOGISTIC REGRESSION")
print(f"{'='*70}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_scaled, y_train)

auc_train_lr = roc_auc_score(y_train, lr.predict_proba(X_train_scaled)[:, 1])
auc_test_lr = roc_auc_score(y_test, lr.predict_proba(X_test_scaled)[:, 1])

print(f"\n✅ Logistic Regression Performance:")
print(f"  Train AUC: {auc_train_lr:.3f}")
print(f"  Test AUC: {auc_test_lr:.3f}")

cv_scores_lr = cross_val_score(
    LogisticRegression(random_state=42, max_iter=1000),
    X_train_scaled, y_train,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc'
)

print(f"  5-Fold CV AUC: {cv_scores_lr.mean():.3f} ± {cv_scores_lr.std():.3f}")

print(f"\nClassification Report (Test Set):")
y_pred_lr = lr.predict(X_test_scaled)
print(classification_report(y_test, y_pred_lr))

# SECTION 4: Random Forest
print(f"\n{'='*70}")
print("MODEL 2: RANDOM FOREST")
print(f"{'='*70}")

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    class_weight='balanced'
)

rf.fit(X_train, y_train)

auc_train_rf = roc_auc_score(y_train, rf.predict_proba(X_train)[:, 1])
auc_test_rf = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])

print(f"\n✅ Random Forest Performance:")
print(f"  Train AUC: {auc_train_rf:.3f}")
print(f"  Test AUC: {auc_test_rf:.3f}")

cv_scores_rf = cross_val_score(
    RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, class_weight='balanced'),
    X_train, y_train,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc'
)

print(f"  5-Fold CV AUC: {cv_scores_rf.mean():.3f} ± {cv_scores_rf.std():.3f}")

# Feature Importance
feature_importance = pd.DataFrame({
    'Feature': ['Competitor Threat', 'Tenure', 'Monthly Spend', 'Total Spend', 'Segment'],
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

print(f"\n✅ Feature Importance (Random Forest):")
for _, row in feature_importance.iterrows():
    print(f"  {row['Feature']:25s}: {row['Importance']*100:5.1f}%")

# SECTION 5: Save Models
with open('logistic_regression_model.pkl', 'wb') as f:
    pickle.dump((lr, scaler), f)

with open('random_forest_model.pkl', 'wb') as f:
    pickle.dump(rf, f)

print(f"\n✅ Models saved")

# Save results
model_results = pd.DataFrame([
    {
        'Model': 'Logistic Regression',
        'Train_AUC': auc_train_lr,
        'Test_AUC': auc_test_lr,
        'CV_Mean': cv_scores_lr.mean(),
        'CV_Std': cv_scores_lr.std()
    },
    {
        'Model': 'Random Forest',
        'Train_AUC': auc_train_rf,
        'Test_AUC': auc_test_rf,
        'CV_Mean': cv_scores_rf.mean(),
        'CV_Std': cv_scores_rf.std()
    }
])

model_results.to_csv('model_performance.csv', index=False)
feature_importance.to_csv('feature_importance.csv', index=False)

print(f"✅ Results saved to CSV")